# Module 2: Building the SciPy Knowledge Base

## Learning Objectives

By the end of this module, you will:
- Scrape and process documentation for RAG
- Implement effective chunking strategies
- Build a production-ready vector store with ChromaDB
- Test retrieval quality

## Setup

In [1]:
import os
import sys
import json
from datetime import datetime, UTC
from dataclasses import asdict
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from dotenv import load_dotenv
load_dotenv()

print(f"OpenAI API Key: {'Set' if os.getenv('OPENAI_API_KEY') else 'Not set'}")

OpenAI API Key: Set


In [2]:
EMBEDDING_MODEL = "text-embedding-3-small"

# If you prefer Chroma to call OpenAI directly, set this to False.
USE_OPENAI_SDK_EMBEDDINGS = True

try:
    from openai import OpenAI
    openai_client = OpenAI()
except Exception as e:
    USE_OPENAI_SDK_EMBEDDINGS = False
    openai_client = None
    print("⚠️ OpenAI SDK not available; falling back to Chroma embedding function.")

## 1. Web Scraping SciPy Documentation

### Scraping Strategy

When scraping documentation, we need to be respectful and efficient:

1. **Respect robots.txt**: Check what's allowed to scrape
2. **Rate limiting**: Don't overwhelm the server (we use 0.5s delay)
3. **User-Agent**: Identify ourselves clearly
4. **Caching**: Save scraped data to avoid re-scraping

Let's look at our scraper implementation.

In [3]:
from scraper import SciPyDocsScraper, create_sample_dataset, ScrapedDocument

print("SciPy modules we can scrape:")
for module in SciPyDocsScraper.MODULES:
    print(f" - scipy.{module}")

SciPy modules we can scrape:
 - scipy.cluster
 - scipy.constants
 - scipy.fft
 - scipy.integrate
 - scipy.interpolate
 - scipy.io
 - scipy.linalg
 - scipy.ndimage
 - scipy.optimize
 - scipy.signal
 - scipy.sparse
 - scipy.spatial
 - scipy.special
 - scipy.stats


### Option A: Use Sample Dataset (Fast, No Network)

### Data freshness and provenance

Vector search quality depends heavily on **what docs you scraped** and **when** you scraped them.

To keep this notebook from going stale, we attach provenance fields to each document (and propagate them into chunk metadata):

- `source_url`: where the text came from (prefer versioned docs URLs when possible)
- `retrieved_at`: ISO timestamp of when you built the dataset (knowledge base)
- `scipy_doc_version`: optional (if you know the doc version you scraped)

If you use **Option A (sample dataset)**, treat it as a *demo corpus*; for a production-ish KB, use **Option B (live scrape)** and store URLs + timestamps.

In [4]:
sample_docs = create_sample_dataset()

print(f"Sample dataset contains {len(sample_docs)} documents")
print("\nSample document:")
print(f"  Title: {sample_docs[0]['title']}")
print(f"  Module: {sample_docs[0]['module']}")
print(f"  Signature: {sample_docs[0]['signature'][:80]}...")

Sample dataset contains 5 documents

Sample document:
  Title: scipy.optimize.minimize
  Module: scipy.optimize
  Signature: scipy.optimize.minimize(fun, x0, args=(), method=None, jac=None, hess=None, hess...


In [5]:
# Save sample dataset
data_dir = Path.cwd().parent / 'data' / 'processed'
data_dir.mkdir(parents=True, exist_ok=True)

with open(data_dir / 'scipy_sample.json', 'w') as f:
    json.dump(sample_docs, f, indent=2)
    
print(f"Saved to {data_dir / 'scipy_sample.json'}")

Saved to /Users/cynthia/Desktop/scipy_rag/data/processed/scipy_sample.json


### Option B: Live Scraping

If you want the full experience, uncomment and run this cell to scrape real documentation.

Freshness tips:
- Prefer scraping **versioned** docs URLs (so the KB can be reproduced later).
- Persist `source_url`, `retrieved_at`, and (if known) `scipy_doc_version` into your dataset so you can refresh intentionally.

In [6]:
# # UNCOMMENT TO SCRAPE LIVE DATA

# scraper = SciPyDocsScraper(
#     delay=0.5, 
#     output_dir=str(Path.cwd().parent / "data" / "raw")
# )

# # Scrape a subset of modules, but can add more if you want
# modules_to_scrape = ["optimize", "integrate"] 
# live_docs = scraper.scrape_all(modules=modules_to_scrape)

# print(f"\nScraped {len(live_docs)} documents from live site")

### Examining the Scraped Data

Let's look at what information we extracted from each documentation page.

In [7]:
# Load our sample dataset
with open(Path.cwd().parent / "data" / "raw" / 'scipy_all.json', 'r') as f:
    documents = json.load(f)

doc = documents[0]

print(f"Function: {doc['title']}")
print(f"\n📦 Module: {doc['module']}")
print(f"📝 Type: {doc['doc_type']}")
print(f"\n📋 Signature:\n{doc['signature']}")
print(f"\n📖 Description:\n{doc['description']}")
print(f"\n🔧 Parameters (first 500 chars):\n{doc['parameters'][:500]}...")
print(f"\n📤 Returns:\n{doc['returns']}")
print(f"\n💻 Examples (first 500 chars):\n{doc['examples'][:500]}...")

Function: show_options#

📦 Module: scipy.optimize
📝 Type: function

📋 Signature:
scipy.optimize.show_options(solver=None,method=None,disp=True)[source]#

📖 Description:
Show documentation for additional options of optimization solvers.

🔧 Parameters (first 500 chars):
...

📤 Returns:


💻 Examples (first 500 chars):
Try it in your browser! We can print documentations of a solver in stdout: >>> from scipy.optimize import show_options >>> show_options ( solver = "minimize" ) ... Specifying a method is possible: >>> show_options ( solver = "minimize" , method = "Nelder-Mead" ) ... We can also get the documentations as a string: >>> show_options ( solver = "minimize" , method = "Nelder-Mead" , disp = False ) Minimization of scalar function of one or more variables using the ...
Go Back Open In Tab
document.addE...


In [8]:
# Attach provenance to documents (keeps the KB auditable + easier to refresh)
now_iso = datetime.now(UTC).isoformat(timespec='seconds').replace("+00:00", "Z")

for scraped_doc in documents:
    scraped_doc.setdefault("retrieved_at", now_iso)
    scraped_doc.setdefault("source_url", None)
    scraped_doc.setdefault("scipy_doc_version", None)

print("Provenance fields ensured on all documents.")

Provenance fields ensured on all documents.


## 2. Document Chunking Strategies

### Why Chunking Matters

Chunking is **critical** for RAG quality. Poor chunking leads to:
- Retrieved context missing key information
- Code examples split in the middle
- Related information separated across chunks

### Chunking Strategies

| Strategy | Pros | Cons | Best For |
|----------|------|------|----------|
| **Fixed-size** | Simple, predictable | May split sentences/code | Uniform text |
| **Recursive** | Respects boundaries | May create uneven chunks | Prose/articles |
| **Code-aware** | Preserves code blocks | More complex | Documentation |
| **Semantic** | Coherent topics | Expensive (needs embeddings) | Long documents |

In [9]:
from chunker import (
    fixed_size_chunker,
    recursive_text_splitter,
    code_aware_chunker,
    chunk_scipy_document,
    chunk_documents,
    Chunk,
    TokenCounter
)

### 2.1 Fixed-Size Chunking

In [10]:
# Sample text with code
sample_text = documents[1]['full_text']  # curve_fit

# Fixed-size chunking
fixed_chunks = fixed_size_chunker(sample_text, chunk_size=150, overlap=50)

print(f"Fixed-size chunking produced {len(fixed_chunks)} chunks")

for i, chunk in enumerate(fixed_chunks[:3]):  # Show first 3
    print(f"\nChunk {i} ({len(chunk)} chars):")
    print("-" * 40)
    print(chunk[:200] + "..." if len(chunk) > 200 else chunk)

Fixed-size chunking produced 24 chunks

Chunk 0 (150 chars):
----------------------------------------
SciPy API Optimization and root finding ( scipy.optimize ) OptimizeResult scipy.optimize. OptimizeResult # class scipy.optimize. OptimizeResult [sourc

Chunk 1 (150 chars):
----------------------------------------
sult # class scipy.optimize. OptimizeResult [source] # Represents the optimization result. Attributes : x ndarray The solution of the optimization. su

Chunk 2 (150 chars):
----------------------------------------
s : x ndarray The solution of the optimization. success bool Whether or not the optimizer exited successfully. status int Termination status of the op


**Problem**: Notice how fixed-size chunking might cut text mid-sentence or mid-code-block!

### 2.2 Recursive Text Splitting

In [11]:
# Recursive chunking. This respects paragraph/sentence boundaries
recursive_chunks = recursive_text_splitter(sample_text, chunk_size=150, overlap=50)

print(f"Recursive chunking produced {len(recursive_chunks)} chunks")

for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"\nChunk {i} ({len(chunk)} chars):")
    print("-" * 40)
    print(chunk[:200] + "..." if len(chunk) > 200 else chunk)

Recursive chunking produced 23 chunks

Chunk 0 (127 chars):
----------------------------------------
SciPy API Optimization and root finding ( scipy.optimize ) OptimizeResult scipy.optimize. OptimizeResult # class scipy.optimize

Chunk 1 (117 chars):
----------------------------------------
OptimizeResult [source] # Represents the optimization result. Attributes : x ndarray The solution of the optimization

Chunk 2 (109 chars):
----------------------------------------
success bool Whether or not the optimizer exited successfully. status int Termination status of the optimizer


**Better**: Recursive splitting tries to keep sentences together!

### 2.3 Code-Aware Chunking

In [12]:
# Code-aware chunking - keeps code blocks intact
code_chunks = code_aware_chunker(sample_text, chunk_size=1000, overlap=50)

print(f"Code-aware chunking produced {len(code_chunks)} chunks")

for i, chunk in enumerate(code_chunks[:3]):
    print(f"\nChunk {i} ({len(chunk)} chars):")
    print("-" * 40)
    # Check if chunk contains code
    has_code = '>>>' in chunk or 'import' in chunk or 'def ' in chunk
    print(f"[Contains code: {has_code}]")
    print(chunk[:950] + "..." if len(chunk) > 950 else chunk)

Code-aware chunking produced 3 chunks

Chunk 0 (928 chars):
----------------------------------------
[Contains code: False]
SciPy API Optimization and root finding ( scipy.optimize ) OptimizeResult scipy.optimize. OptimizeResult # class scipy.optimize. OptimizeResult [source] # Represents the optimization result. Attributes : x ndarray The solution of the optimization. success bool Whether or not the optimizer exited successfully. status int Termination status of the optimizer. Its value depends on the underlying solver. Refer to message for details. message str Description of the cause of the termination. fun float Value of objective function at x . jac, hess ndarray Values of objective functionâs Jacobian and its Hessian at x (if available). The Hessian may be an approximation, see the documentation of the function in question. hess_inv object Inverse of the objective functionâs Hessian; may be an approximation. Not available for all solvers. The type of this attribute may be eit

**Best for docs**: Code-aware chunking prioritizes keeping code examples complete!

### 2.4 Structured Document Chunking

For SciPy docs, we use a special strategy that creates multiple chunk types:
1. **Summary chunk**: Signature + description + key parameters
2. **Examples chunk**: Code examples (kept together)
3. **Full-text chunks**: Remaining content

In [13]:
# Chunk a single document with our specialized function
doc_chunks = chunk_scipy_document(documents[0], strategy="code_aware", chunk_size=600)

print(f"Document '{documents[0]['title']}' produced {len(doc_chunks)} chunks:")
print("\n" + "="*60)

for chunk in doc_chunks:
    print(f"\n📄 {chunk.chunk_id}")
    print(f"   Type: {chunk.metadata['chunk_type']}")
    print(f"   Size: {len(chunk.text)} chars")
    print(f"   Preview: {chunk.text[:100]}...")

Document 'show_options#' produced 6 chunks:


📄 show_options_summary
   Type: summary
   Size: 139 chars
   Preview: scipy.optimize.show_options(solver=None,method=None,disp=True)[source]#

Show documentation for addi...

📄 show_options_examples
   Type: examples
   Size: 632 chars
   Preview: Examples for show_options:

Try it in your browser! We can print documentations of a solver in stdou...

📄 show_options_full_0
   Type: full_text
   Size: 591 chars
   Preview: SciPy API Optimization and root finding ( scipy.optimize ) show_options scipy.optimize. show_options...

📄 show_options_full_1
   Type: full_text
   Size: 236 chars
   Preview: Otherwise, show only the options for the specified method. Valid values corresponds to methodsâ na...

📄 show_options_full_2
   Type: full_text
   Size: 597 chars
   Preview: Returns : text Either None (for disp=True) or the text string (disp=False) Notes The solver-specific...

📄 show_options_full_3
   Type: full_text
   Size: 599 chars
   Previe

### 2.5 Chunking All Documents

In [14]:
# Chunk all documents
all_chunks = chunk_documents(documents, strategy="code_aware", chunk_size=600, overlap=100)

print(f"Total chunks created: {len(all_chunks)}")

# Statistics
chunk_types = {}
chunk_sizes = [len(c.text) for c in all_chunks]

for chunk in all_chunks:
    ct = chunk.metadata['chunk_type']
    chunk_types[ct] = chunk_types.get(ct, 0) + 1

print(f"\nChunk type distribution:")
for ct, count in chunk_types.items():
    print(f"  {ct}: {count}")

print(f"\nChunk size statistics:")
print(f"  Min: {min(chunk_sizes)} chars")
print(f"  Max: {max(chunk_sizes)} chars")
print(f"  Avg: {sum(chunk_sizes) / len(chunk_sizes):.0f} chars")

Total chunks created: 7728

Chunk type distribution:
  summary: 799
  examples: 676
  full_text: 6253

Chunk size statistics:
  Min: 14 chars
  Max: 8696 chars
  Avg: 534 chars


In [15]:
# Propagate document provenance into chunk metadata (helps audits + refreshes)
by_title = {d.get("title"): d for d in documents}

for ch in all_chunks:
    fn = ch.metadata.get("function_name")
    doc = by_title.get(fn)
    if doc is None:
        continue

    ch.metadata.setdefault("source_url", doc.get("source_url"))
    ch.metadata.setdefault("retrieved_at", doc.get("retrieved_at"))
    ch.metadata.setdefault("scipy_doc_version", doc.get("scipy_doc_version"))

print("Chunk metadata updated with provenance where available.")

Chunk metadata updated with provenance where available.


## 3. Building the Vector Store

Now let's embed our chunks and store them in ChromaDB.

In [16]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

chroma_path = Path.cwd().parent / "chroma_db"
chroma_path.mkdir(exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(chroma_path))
print(f"ChromaDB initialized at {chroma_path}")

ChromaDB initialized at /Users/cynthia/Desktop/scipy_rag/chroma_db


In [17]:
# Create / reset the collection
try:
    chroma_client.delete_collection("scipy_docs")
    print("Deleted existing collection")
except Exception:
    pass

collection_kwargs = {
    "name": "scipy_docs",
    "metadata": {
        "description": "SciPy documentation for RAG",
        "hnsw:space": "cosine"  # Use cosine similarity
    }
}

if USE_OPENAI_SDK_EMBEDDINGS:
    # We'll supply embeddings ourselves via the OpenAI SDK.
    collection = chroma_client.create_collection(**collection_kwargs)
else:
    # Let Chroma call OpenAI for embeddings (convenient, but less flexible).
    openai_ef = OpenAIEmbeddingFunction(
        api_keys=os.getenv("OPEN_API_KEY"),
        model_name=EMBEDDING_MODEL
    )
    collection = chroma_client.create_collection(
        **collection_kwargs,
        embedding_function=openai_ef
    )

print(f"Created collection: {collection.name}")
print(f"Embedding mode: {'OpenAI SDK' if USE_OPENAI_SDK_EMBEDDINGS else 'Chroma embedding function'}")
print(f"Embedding model: {EMBEDDING_MODEL}")

Deleted existing collection
Created collection: scipy_docs
Embedding mode: OpenAI SDK
Embedding model: text-embedding-3-small


In [18]:
def embed_texts(texts):
    # Embed a list of strings using the configured embedding model.
    # Returns: List[List[float]] embeddings
    if not USE_OPENAI_SDK_EMBEDDINGS:
        raise RuntimeError("embed_texts called but USE_OPENAI_SDK_EMBEDDINGS is False")

    if openai_client is None:
        raise RuntimeError("OpenAI client not initialized")

    resp = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [item.embedding for item in resp.data]

In [19]:
def clean_metadata(metadata):
    return { key: "" if value is None else value for key, value in metadata.items()}

In [20]:
from tqdm import tqdm

# Add chunks to collection in batches
batch_size = 50

for i in tqdm(range(0, len(all_chunks), batch_size), desc="Indexing chunks"):
    batch = all_chunks[i:i + batch_size]

    ids = [chunk.chunk_id for chunk in batch]
    docs = [chunk.text for chunk in batch]
    metas = [clean_metadata(chunk.metadata) for chunk in batch]

    if USE_OPENAI_SDK_EMBEDDINGS:
        embs = embed_texts(docs)
        collection.add(
            ids=ids,
            documents=docs,
            metadatas=metas,
            embeddings=embs
        )
    else:
        collection.add(
            ids=ids,
            documents=docs,
            metadatas=metas
        )

print(f"\n✅ Added {collection.count()} chunks to the vector store")

Indexing chunks: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 155/155 [01:37<00:00,  1.58it/s]


✅ Added 7698 chunks to the vector store


## 4. Retrieval Testing

Let's test our knowledge base with various queries!

In [21]:
def search_scipy(query: str, n_results: int = 3, filter_module: str = None):
    """Search the SciPy knowledge base."""
    where = {"module": filter_module} if filter_module else None

    if USE_OPENAI_SDK_EMBEDDINGS:
        query_emb = embed_texts([query])[0]
        results = collection.query(
            query_embeddings=[query_emb],
            n_results=n_results,
            where=where
        )
    else:
        results = collection.query(
            query_texts=[query],
            n_results=n_results,
            where=where
        )

    print(f"🔍 Query: '{query}'")
    if filter_module:
        print(f"   Filter: {filter_module}")
    print("=" * 60)

    for i, (doc, metadata, distance) in enumerate(
        zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        )
    ):
        fn = metadata.get("function_name", "unknown")
        ct = metadata.get("chunk_type", "chunk")
        mod = metadata.get("module", "unknown")
        print(f"\n[{i + 1}] {fn} ({ct})")
        print(f"    Module: {mod}")
        print(f"    Distance: {distance:.4f}")
        if metadata.get("source_url"):
            print(f"    Source: {metadata['source_url']}")
        if metadata.get("retrieved_at"):
            print(f"    Retrieved: {metadata['retrieved_at']}")
        print(f"    Preview: {doc[:150]}...")

In [22]:
search_scipy("How do I fit a curve to my experimental data?")

🔍 Query: 'How do I fit a curve to my experimental data?'

[1] curve_fit (examples)
    Module: scipy.optimize
    Distance: 0.4884
    Preview: Examples for curve_fit:

Try it in your browser! >>> import numpy as np >>> import matplotlib.pyplot as plt >>> from scipy.optimize import curve_fit >...

[2] curve_fit (full_text)
    Module: scipy.optimize
    Distance: 0.4915
    Retrieved: 2026-06-08T19:01:20.537862+00:00
    Preview: ```
>>> popt, pcov = curve_fit(func, xdata, ydata, bounds=(0, [3., 1., 0.5]))
>>> popt
array([2.43736712, 1.        , 0.34463856])
>>> plt.plot(xdata,...

[3] curve_fit (full_text)
    Module: scipy.optimize
    Distance: 0.5099
    Retrieved: 2026-06-08T19:01:20.537862+00:00
    Preview: Water Resources Research, Vol. 43, W03423, DOI:10.1029/2005WR004804 Examples Try it in your browser! ```
>>> import numpy as np
>>> import matplotlib....


In [23]:
search_scipy("calculate the integral of a function")

🔍 Query: 'calculate the integral of a function'

[1] quad (summary)
    Module: scipy.integrate
    Distance: 0.4279
    Retrieved: 2026-06-08T19:01:51.744808+00:00
    Preview: scipy.integrate.quad(func,a,b,args=(),full_output=0,epsabs=1.49e-08,epsrel=1.49e-08,limit=50,points=None,weight=None,wvar=None,wopts=None,maxp1=50,lim...

[2] tplquad (full_text)
    Module: scipy.integrate
    Distance: 0.4284
    Retrieved: 2026-06-08T19:01:54.037557+00:00
    Preview: Calculate \(\int^{x=1}_{x=0} \int^{y=1}_{y=0} \int^{z=1}_{z=0} a x y z \,dz \,dy \,dx\) for \(a=1, 3\) . 
```
>>> f = lambda z, y, x, a: a*x*y*z
>>> i...

[3] dblquad (full_text)
    Module: scipy.integrate
    Distance: 0.4478
    Retrieved: 2026-06-08T19:01:53.465704+00:00
    Preview: ```
>>> f = lambda y, x, a: a*x*y
>>> integrate.dblquad(f, 0, 1, lambda x: x, lambda x: 2-x, args=(1,))
    (0.33333333333333337, 5.551115123125783e-1...


#### Out-of-corpus query

In [24]:
search_scipy("solve a system of linear equations")

🔍 Query: 'solve a system of linear equations'

[1] lu_solve (summary)
    Module: scipy.linalg
    Distance: 0.4979
    Retrieved: 2026-06-08T19:02:31.362440+00:00
    Preview: scipy.linalg.lu_solve(lu_and_piv,b,trans=0,overwrite_b=False,check_finite=True)[source]#

Solve an equation system, a x = b, given the LU factorizatio...

[2] solve (summary)
    Module: scipy.linalg
    Distance: 0.5227
    Retrieved: 2026-06-08T19:02:14.028059+00:00
    Preview: scipy.linalg.solve(a,b,lower=False,overwrite_a=False,overwrite_b=False,check_finite=True,assume_a=None,transposed=False)[source]#

Solve the equationa...

[3] newton_krylov (full_text)
    Module: scipy.optimize
    Distance: 0.5310
    Retrieved: 2026-06-08T19:01:46.697794+00:00
    Preview: DOI:10.1137/1.9780898718898.ch3 [ 2 ] D.A. Knoll and D.E. Keyes, J. Comp. Phys. 193, 357 (2004). DOI:10.1016/j.jcp.2003.08.010 [ 3 ] A.H. Baker and E....


In [25]:
search_scipy("filter a signal with butterworth")

🔍 Query: 'filter a signal with butterworth'

[1] butter (examples)
    Module: scipy.signal
    Distance: 0.3551
    Preview: Examples for butter:

Try it in your browser! Design an analog filter and plot its frequency response, showing the critical points: >>> from scipy imp...

[2] filtfilt (full_text)
    Module: scipy.signal
    Distance: 0.4026
    Retrieved: 2026-06-08T19:03:23.522832+00:00
    Preview: ```
>>> b, a = signal.butter(8, 0.125)
>>> y = signal.filtfilt(b, a, x, padlen=150)
>>> np.abs(y - xlow).max()
9.1086182074789912e-06


```
 We get a ...

[3] butter (full_text)
    Module: scipy.signal
    Distance: 0.4115
    Retrieved: 2026-06-08T19:03:56.983534+00:00
    Preview: ```
>>> sos = signal.butter(10, 15, 'hp', fs=1000, output='sos')
>>> filtered = signal.sosfilt(sos, sig)
>>> ax2.plot(t, filtered)
>>> ax2.set_title('...


In [26]:
# Test with module filter
search_scipy("optimization", n_results=2, filter_module="scipy.optimize")

🔍 Query: 'optimization'
   Filter: scipy.optimize

[1] fmin_cobyla (full_text)
    Module: scipy.optimize
    Distance: 0.5170
    Retrieved: 2026-06-08T19:01:40.934916+00:00
    Preview: References Powell M.J.D. (1994), âA direct search optimization method that models the objective and constraint functions by linear interpolation.â...

[2] basinhopping (full_text)
    Module: scipy.optimize
    Distance: 0.5203
    Retrieved: 2026-06-08T19:01:14.534545+00:00
    Preview: Default is 0.9. Added in version 1.8.0. Returns : res OptimizeResult The optimization result represented as a OptimizeResult object. Important attribu...


### Debugging Poor Retrieval Results

If retrieval quality is poor, common issues include:

1. **Poor chunking**: Chunks are too small, too large, or split important examples
2. **Missing metadata**: Cannot filter or rerank by module, function, doc type, etc.
3. **Corpus coverage gap**: The right answer is not in the indexed corpus
4. **Intent mismatch**: Retrieved docs are semantically similar but answer the wrong question
5. **Weak ranking**: Broad queries do not surface the best overview or entrypoint docs
6. **Poor text cleaning**: Relevant chunks have broken spacing, code formatting, or encoding artifacts
7. **No fallback behavior**: System answers from weak evidence instead of saying the corpus is missing relevant docs

In [27]:
# Check what chunks contain for a specific function
def inspect_function_chunks(function_name: str):
    """See all chunks for a specific function."""
    results = collection.get(
        where={"function_name": function_name}
    )
    
    print(f"Chunks for '{function_name}':")
    print("⚙️" * 60)
    
    for i, (doc_id, doc, metadata) in enumerate(zip(
        results['ids'],
        results['documents'],
        results['metadatas']
    )):
        print(f"\n[{i+1}] {doc_id}")
        print(f"    Type: {metadata['chunk_type']}")
        print(f"    Size: {len(doc)} chars")
        print(f"    Content preview:")
        print(f"    {doc[:200]}...")

inspect_function_chunks("curve_fit")

Chunks for 'curve_fit':
⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️⚙️

[1] curve_fit_summary
    Type: summary
    Size: 251 chars
    Content preview:
    scipy.optimize.curve_fit(f,xdata,ydata,p0=None,sigma=None,absolute_sigma=False,check_finite=None,bounds=(-inf,inf),method=None,jac=None,*,full_output=False,nan_policy=None,**kwargs)[source]#

Use non-...

[2] curve_fit_examples
    Type: examples
    Size: 3247 chars
    Content preview:
    Examples for curve_fit:

Try it in your browser! >>> import numpy as np >>> import matplotlib.pyplot as plt >>> from scipy.optimize import curve_fit >>> def func ( x , a , b , c ): ... return a * np ....

[3] curve_fit_full_0
    Type: full_text
    Size: 495 chars
    Content preview:
    SciPy API Optimization and root finding ( scipy.optimize ) curve_fit scipy.optimize. curve_fit # scipy.optimize. curve_fit ( f , xdata , ydata , p0 = None , sigma = None , absolute_sigm

## Module 2 Summary

### What We Built

1. **Scraper**: Can fetch live SciPy documentation (or use sample data)
2. **Chunker**: Multiple strategies with code-awareness
3. **Vector Store**: Persistent ChromaDB with metadata

### Key Takeaways

- **Scraping**: Consider rate limiting, user-agent, caching
- **Chunking**: Use code-aware strategies for documentation
- **Metadata**: Essential for filtering and debugging
- **Testing**: Always validate retrieval quality

### Next Module

In **Module 3**, we'll build the complete RAG pipeline:
- Retrieval logic with preprocessing
- Prompt engineering for code generation
- Integration with OpenAI and Ollama

## Exercises

### Exercise 1: Compare Chunking Strategies

**Goal**: See how different chunkers affect the retrieved text. Which method preserves the most useful context?

In [ ]:
sample_text = documents[1]["full_text"]

fixed_chunks = fixed_size_chunker(sample_text, chunk_size=600, overlap=100)
recursive_chunks = recursive_text_splitter(sample_text, chunk_size=600, overlap=100)
code_chunks = code_aware_chunker(sample_text, chunk_size=600, overlap=100)

print("Fixed:", len(fixed_chunks), "chunks")
print("Recursive:", len(recursive_chunks), "chunks")
print("Code-aware:", len(code_chunks), "chunks")

**Questions to consider:**
1. Which chunk ends mid-sentence or mid-word?
2. Which chunk preserves the most complete thought?
3. For RAG on code documentation, which would you choose?

---

### Exercise 2: Test Retrieval with Your Own Queries

Use `search_scipy()` to test retrieval. Try these queries and examine the results:

In [ ]:
search_scipy("How do I fit a curve to noisy data?", n_results=3)

In [ ]:
search_scipy("How do I compute a definite integral?", n_results=3)

In [ ]:
search_scipy("How do I generate random samples from a normal distribution?", n_results=3)

**Try your own query!** Think of a SciPy task and see what gets retrieved.

In [ ]:
# search_scipy()

In [ ]:
# search_scipy()

In [ ]:
# search_scipy()

### Exercise 3: Faceted Search with Metadata Filters

**Goal**: See how filtering by module improves retrieval for broad queries.

In [ ]:
# Broad query without filter
search_scipy("optimization", n_results=5)

In [ ]:
# Same query with module filter
search_scipy("optimization", n_results=5, filter_module="scipy.optimize")

**Questions:**
1. How did filtering by module change the results?
2. When would an unfiltered search be better?
3. Try filtering with `filter_module="scipy.integrate"` or `"scipy.signal"`.

### Exercise 4: Experiment with Chunk Sizes

Try different chunk sizes (400, 800, 1200) and compare retrieval quality.

### Exercise 5: Add More Modules

Scrape additional SciPy modules and add them to the vector store.



### Exercise 6: Add Custom Metadata

Extend the metadata schema to include fields like `difficulty_level`, `related_functions`, or `use_case_tags`.